In [ ]:
import xarray as xr
import pandas as pd
import numpy as np
import datetime as dt
import os
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap, Normalize
from matplotlib.dates import DateFormatter
from matplotlib.lines import Line2D
from matplotlib.patches import Rectangle, Patch
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import cartopy.feature.nightshade as nightshade
import pysolar.solar as solar

# ============= НАСТРОЙКИ ШРИФТОВ =============
plt.rcParams['xtick.labelsize'] = 20    # Цифры на оси X
plt.rcParams['ytick.labelsize'] = 28    # Цифры на оси Y
plt.rcParams['legend.fontsize'] = 16    # Легенда

# ============= ПАРАМЕТРЫ =============
file_year = 2015
altitude_tolerance = 0.5  # Допустимое отклонение от целевой высоты в км

temp_min_loc = 215
temp_max_loc = 225
temp_min_glob = 185
temp_max_glob = 245

# ============= НАСТРОЙКИ ТОЧЕК ПРОФИЛЕЙ =============
profile_point_mode = 'single'  # 'single' или 'range'
single_point_height = 17.5
single_median_max = 25.0
range_min = 90.0
range_max = 110.0

# Пути к файлам
hdf_matching_path = f'C:/Users/Maks/Desktop/Jupyter/output_full_saber_data/{file_year}_saber_overpass_matching.h5'
thunderbolts_path = f'C:/Users/Maks/Desktop/Jupyter/thunderbolts_data/hdf/{file_year}_thunderbolts_clastered.h5'
base_satellite_directory = 'G:/SABER_L2A'
output_plot_dir = f'C:/Users/Maks/Desktop/Jupyter/temp_profiles/2/10-25км/{file_year}'

# Создаем папку для сохранения
os.makedirs(output_plot_dir, exist_ok=True)

# ============= ФУНКЦИИ =============
def yyddd_to_date(yyddd):
    yyddd_str = str(int(yyddd))
    year = int(yyddd_str[:4])
    day_of_year = int(yyddd_str[4:])
    return dt.datetime(year, 1, 1) + dt.timedelta(days=day_of_year - 1)

def calculate_solar_position(utc_time):
    if utc_time.tzinfo is None:
        utc_time = utc_time.replace(tzinfo=dt.timezone.utc)
    
    best_lat = 0
    best_lon = 0
    max_alt = -90
    
    for lat in range(-90, 91, 10):
        for lon in range(-180, 180, 10):
            try:
                alt = solar.get_altitude(float(lat), float(lon), utc_time)
                if alt > max_alt:
                    max_alt = alt
                    best_lat = lat
                    best_lon = lon
            except Exception:
                continue
    
    for lat in np.arange(best_lat - 5, best_lat + 5, 0.5):
        for lon in np.arange(best_lon - 5, best_lon + 5, 0.5):
            try:
                alt = solar.get_altitude(float(lat), float(lon), utc_time)
                if alt > max_alt:
                    max_alt = alt
                    best_lat = lat
                    best_lon = lon
            except Exception:
                continue
    
    sun_lon = best_lon if best_lon >= 0 else best_lon + 360
    sun_lat = best_lat
    return sun_lon, sun_lat

def get_illumination_category(sun_lon, sun_lat, point_lon, point_lat, margin=10):
    sun_lon_rad = np.radians(sun_lon if sun_lon <= 180 else sun_lon - 360)
    sun_lat_rad = np.radians(sun_lat)
    
    if point_lon > 180:
        point_lon_rad = np.radians(point_lon - 360)
    else:
        point_lon_rad = np.radians(point_lon)
    point_lat_rad = np.radians(point_lat)
    
    cos_zenith = (np.sin(sun_lat_rad) * np.sin(point_lat_rad) + 
                  np.cos(sun_lat_rad) * np.cos(point_lat_rad) * 
                  np.cos(point_lon_rad - sun_lon_rad))
    
    cos_zenith = np.clip(cos_zenith, -1, 1)
    zenith_angle = np.degrees(np.arccos(cos_zenith))
    
    if zenith_angle < 90 - margin:
        return 'day'
    elif zenith_angle > 90 + margin:
        return 'night'
    else:
        return 'terminator'

# ============= ЗАГРУЗКА ДАННЫХ =============
print("Загрузка данных...")
thunderbolts_df = pd.read_hdf(thunderbolts_path, key='strikes')
satellite_overpass_matching = pd.read_hdf(hdf_matching_path, key='matches')

# Получаем список всех кластеров
all_clusters = satellite_overpass_matching.index.tolist()
print(f"Найдено кластеров для обработки: {len(all_clusters)}")
print(f"Список кластеров: {all_clusters}\n")

colors = [(0.5, 0.0, 0.5), (0, 0, 1), (0, 0.5, 1), (0.5,1,0.5), 
          (1, 1, 0), (1, 0.5, 0), (1, 0, 0), (0.5, 0, 0)]
custom_cmap = LinearSegmentedColormap.from_list("custom_coolwarm", colors, N=100)

# ============= ОБРАБОТКА ВСЕХ КЛАСТЕРОВ =============
successful_clusters = []
failed_clusters = []

for idx, cluster_num in enumerate(all_clusters, 1):
    try:
        print(f"\n{'='*60}")
        print(f"ОБРАБОТКА КЛАСТЕРА #{cluster_num} ({idx}/{len(all_clusters)})")
        print(f"{'='*60}")
        
        data_row = satellite_overpass_matching.loc[cluster_num]
        
        # ===== ИЗВЛЕКАЕМ ДАННЫЕ ИЗ ROW_DATA =====
        saber_file_val = data_row['satellite_file_name']
        if isinstance(saber_file_val, pd.Series):
            saber_file = str(saber_file_val.iloc[0]).replace('.nc', '')
        else:
            saber_file = str(saber_file_val).replace('.nc', '')
        
        altitude_val = data_row['altitude_sat']
        if isinstance(altitude_val, pd.Series):
            target_altitude = round(float(altitude_val.iloc[0]), 1)
        else:
            target_altitude = round(float(altitude_val), 1)
        
        time_val = data_row['matched_time']
        if isinstance(time_val, pd.Series):
            matched_time = pd.to_datetime(time_val.iloc[0])
        else:
            matched_time = pd.to_datetime(time_val)
        
        flash_val = data_row['flash_time']
        if isinstance(flash_val, pd.Series):
            last_flash_time = pd.to_datetime(flash_val.iloc[0])
        else:
            last_flash_time = pd.to_datetime(flash_val)
        
        amp_val = data_row['flash_amp']
        if isinstance(amp_val, pd.Series):
            flash_amp = float(amp_val.iloc[0])
        else:
            flash_amp = float(amp_val)
        
        file_parts = saber_file.split('_')
        yyddd = file_parts[2]
        day = yyddd[-3:]
        
        possible_paths = [
            os.path.join(base_satellite_directory, str(file_year), day, f"{saber_file}.nc"),
            os.path.join(base_satellite_directory, str(file_year), f"{saber_file}.nc"),
        ]
        
        nc_file_path = None
        for path in possible_paths:
            if os.path.exists(path):
                nc_file_path = path
                print(f"Найден файл: {path}")
                break
        
        if nc_file_path is None:
            raise FileNotFoundError(f"Файл не найден: {saber_file}.nc")
        
        print(f"Целевая высота: {target_altitude} км")
        print(f"Время совпадения: {matched_time}")
        print(f"Время разряда: {last_flash_time}")
        
        def get_scalar(val):
            if isinstance(val, pd.Series):
                return float(val.iloc[0])
            return float(val)
        
        # ===== РАЗРЯДЫ КЛАСТЕРА =====
        cluster_thunderbolts = thunderbolts_df[thunderbolts_df['clnb'] == cluster_num]
        
        flash_matches = cluster_thunderbolts[cluster_thunderbolts.index == last_flash_time]
        
        if len(flash_matches) > 0:
            if len(flash_matches) == 1:
                thunderbolt = flash_matches.iloc[0]
                flash_lat = get_scalar(thunderbolt['lat'])
                flash_lon = get_scalar(thunderbolt['lon'])
                flash_amp_value = get_scalar(thunderbolt['amp'])
            else:
                max_amp_idx = flash_matches['amp'].abs().idxmax()
                flash_lat = get_scalar(flash_matches.loc[max_amp_idx, 'lat'])
                flash_lon = get_scalar(flash_matches.loc[max_amp_idx, 'lon'])
                flash_amp_value = get_scalar(flash_matches.loc[max_amp_idx, 'amp'])
            
            print(f"Найден разряд: ({flash_lat:.3f}°, {flash_lon:.3f}°) мощностью {flash_amp_value:.0f} A")
        else:
            print(f"Разряд не найден!")
            flash_lat, flash_lon, flash_amp_value = None, None, None
        
        # ===== ЗАГРУЗКА ВСЕХ ДАННЫХ СПУТНИКА =====
        print(f"\nЗагрузка данных из файла...")
        
        ds = xr.open_dataset(nc_file_path)
        
        time_data = ds['time'].values
        ktemp_data = ds['ktemp'].values
        lat_data = ds['tplatitude'].values
        lon_data = ds['tplongitude'].values
        alt_data = ds['tpaltitude'].values
        date_value = yyddd_to_date(ds['date'].values[0])
        
        print(f"Дата: {date_value.strftime('%d.%m.%Y')}")
        print(f"Профилей: {len(time_data)}, высотных уровней: {len(ds['altitude'])}")
        
        ds.close()
        
        all_profiles_list = []
        
        for row_idx in range(len(time_data)):
            temps = ktemp_data[row_idx]
            alts = alt_data[row_idx]
            lats = lat_data[row_idx]
            lons = lon_data[row_idx]
            times = time_data[row_idx]
            
            valid_mask = ~np.isnan(temps)
            if not valid_mask.any():
                continue
            
            valid_indices = np.where(valid_mask)[0]
            
            for idx in valid_indices:
                time_ms = float(times[idx])
                point_time = date_value + dt.timedelta(milliseconds=time_ms)
                point_time = pd.Timestamp(point_time).floor('s')
                
                all_profiles_list.append({
                    'datetime': point_time,
                    'altitude_km': float(alts[idx]),
                    'temperature_K': float(temps[idx]),
                    'lat': float(lats[idx]),
                    'lon': float(lons[idx]),
                })
        
        all_profiles_df = pd.DataFrame(all_profiles_list)
        all_profiles_df = all_profiles_df.sort_values(['datetime', 'altitude_km']).reset_index(drop=True)
        
        print(f"Загружено точек: {len(all_profiles_df)}")
        print(f"Диапазон времени: {all_profiles_df['datetime'].min()} - {all_profiles_df['datetime'].max()}")
        
        # ===== ФИЛЬТРАЦИЯ ПО ВЫСОТЕ =====
        all_profiles_df['alt_diff'] = abs(all_profiles_df['altitude_km'] - target_altitude)
        satellite_df = all_profiles_df[all_profiles_df['alt_diff'] <= altitude_tolerance].copy()
        
        if len(satellite_df) == 0:
            closest_points = all_profiles_df.loc[all_profiles_df.groupby('datetime')['alt_diff'].idxmin()]
            satellite_df = closest_points.copy()
            print(f"\nИспользуются ближайшие по высоте точки")
        
        satellite_df['time_diff'] = abs(satellite_df['datetime'] - matched_time)
        match_point = satellite_df.loc[satellite_df['time_diff'].idxmin()].copy()
        
        print(f"\nТочка совпадения: {match_point['datetime']}, высота {match_point['altitude_km']:.1f} км")
        
        traj_df = satellite_df.sort_values('datetime').copy()
        print(f"Точек в траектории: {len(traj_df)}")
        
        flash_times_all = cluster_thunderbolts[cluster_thunderbolts.index < last_flash_time]
        
        time_window_start = matched_time - dt.timedelta(minutes=15)
        flash_times_15min = cluster_thunderbolts[
            (cluster_thunderbolts.index >= time_window_start) & 
            (cluster_thunderbolts.index < last_flash_time)
        ]
        print(f"Разрядов за 15 мин до попадания: {len(flash_times_15min)}")
        
        sun_lon, sun_lat = calculate_solar_position(matched_time)
        illumination = get_illumination_category(sun_lon, sun_lat, match_point['lon'], match_point['lat'])
        flash_sign = 'positive' if flash_amp_value > 0 else 'negative'
        altitude_rounded = int(match_point['altitude_km'])
        
        # ===== ТОЧКИ ПРОФИЛЕЙ =====
        print(f"\nВычисление точек профилей...")
        
        if profile_point_mode == 'single':
            target_point_height = single_point_height
            median_max = single_median_max
            print(f"Режим: одна высота {target_point_height} км, медиана 0-{median_max} км")
        else:  # range mode
            target_point_height = (range_min + range_max) / 2
            print(f"Режим: диапазон {range_min}-{range_max} км")
            print(f"Точки на средней высоте: {target_point_height:.1f} км")
            print(f"Медиана по диапазону {range_min}-{range_max} км")
        
        profile_points_list = []
        
        for dt_time in all_profiles_df['datetime'].unique():
            profile_data = all_profiles_df[all_profiles_df['datetime'] == dt_time]
            
            if profile_point_mode == 'single':
                height_diff = abs(profile_data['altitude_km'] - target_point_height)
                if height_diff.min() <= 1.0:
                    point_idx = height_diff.idxmin()
                    point_lat = profile_data.loc[point_idx, 'lat']
                    point_lon = profile_data.loc[point_idx, 'lon']
                    temps_range = profile_data[profile_data['altitude_km'] <= median_max]['temperature_K']
                else:
                    continue
            else:
                height_diff = abs(profile_data['altitude_km'] - target_point_height)
                if height_diff.min() <= 2.0:
                    point_idx = height_diff.idxmin()
                    point_lat = profile_data.loc[point_idx, 'lat']
                    point_lon = profile_data.loc[point_idx, 'lon']
                    temps_range = profile_data[
                        (profile_data['altitude_km'] >= range_min) & 
                        (profile_data['altitude_km'] <= range_max)
                    ]['temperature_K']
                else:
                    continue
            
            if len(temps_range) > 0:
                median_temp = temps_range.median()
                profile_points_list.append({
                    'datetime': dt_time,
                    'lat': point_lat,
                    'lon': point_lon,
                    'median_temp': median_temp,
                    'altitude': target_point_height
                })
        
        profile_points_df = pd.DataFrame(profile_points_list)
        print(f"Найдено точек профилей: {len(profile_points_df)}")
        if len(profile_points_df) > 0:
            print(f"Диапазон температур: {profile_points_df['median_temp'].min():.1f} - {profile_points_df['median_temp'].max():.1f} K")
        
        # ===== СОЗДАНИЕ ГРАФИКА =====
        print(f"\nСоздание графика...")
        
        fig = plt.figure(figsize=(26, 16))
        gs = fig.add_gridspec(2, 2, width_ratios=[0.4, 0.6], wspace=0.15, hspace=0.4)
        gs_right = gs[:, 1].subgridspec(2, 1, hspace=0.45)
        
        # ----- ЛЕВЫЕ ГРАФИКИ: ПРОФИЛИ -----
        ax1_top = plt.subplot(gs[0, 0])
        
        scatter_top = ax1_top.scatter(all_profiles_df['datetime'], 
                                     all_profiles_df['altitude_km'], 
                                     c=all_profiles_df['temperature_K'], 
                                     cmap=custom_cmap, s=20, alpha=0.7,
                                     vmin=temp_min_glob, vmax=temp_max_glob)
        
        ax1_top.axvline(x=last_flash_time, color='black', linestyle='-', linewidth=2, alpha=0.7,
                       label=f'Разряд: {last_flash_time.strftime("%H:%M:%S")}')
        ax1_top.axvline(x=matched_time, color='gray', linestyle='--', linewidth=2, alpha=0.6,
                       label=f'Попадание: {matched_time.strftime("%H:%M:%S")}')
        
        ax1_top.xaxis.set_major_formatter(DateFormatter('%H:%M:%S'))
        ax1_top.xaxis.set_major_locator(plt.MaxNLocator(8))
        ax1_top.tick_params(axis='x', rotation=45)
        ax1_top.grid(True, alpha=0.3)
        ax1_top.set_xlabel('Время (UTC)', fontsize=26)
        ax1_top.set_ylabel('Высота (км)', fontsize=26)
        ax1_top.set_ylim(0, 120)
        ax1_top.set_title(f'Все профили температуры', fontsize=30, fontweight='bold')
        ax1_top.legend(loc='upper right')
        
        cbar_top = plt.colorbar(scatter_top, ax=ax1_top)
        cbar_top.set_label('Температура (K)', fontsize=24)
        
        # Нижний левый график - 15-минутное окно
        ax1_bottom = plt.subplot(gs[1, 0])
        
        time_window_start = matched_time - dt.timedelta(minutes=15)
        time_window_end = matched_time + dt.timedelta(minutes=15)
        
        window_profiles_df = all_profiles_df[
            (all_profiles_df['datetime'] >= time_window_start) & 
            (all_profiles_df['datetime'] <= time_window_end)
        ].copy()
        
        scatter_bottom = ax1_bottom.scatter(window_profiles_df['datetime'], 
                                           window_profiles_df['altitude_km'], 
                                           c=window_profiles_df['temperature_K'], 
                                           cmap=custom_cmap, s=120, alpha=0.8,
                                           vmin=temp_min_loc, vmax=temp_max_loc)
        
        ax1_bottom.axvline(x=last_flash_time, color='black', linestyle='-', linewidth=2, alpha=0.7,
                          label=f'Разряд: {last_flash_time.strftime("%H:%M:%S")}')
        ax1_bottom.axvline(x=matched_time, color='gray', linestyle='--', linewidth=2, alpha=0.6,
                          label=f'Попадание: {matched_time.strftime("%H:%M:%S")}')
        
        ax1_bottom.xaxis.set_major_formatter(DateFormatter('%H:%M:%S'))
        ax1_bottom.xaxis.set_major_locator(plt.MaxNLocator(8))
        ax1_bottom.tick_params(axis='x', rotation=45)
        ax1_bottom.grid(True, alpha=0.3)
        ax1_bottom.set_xlabel('Время (UTC)', fontsize=26)
        ax1_bottom.set_ylabel('Высота (км)', fontsize=26)
        ax1_bottom.set_ylim(10, 25)
        ax1_bottom.set_title(f'Профили температуры (15 мин)', fontsize=30, fontweight='bold')
        ax1_bottom.legend(loc='upper right')
        
        cbar_bottom = plt.colorbar(scatter_bottom, ax=ax1_bottom)
        cbar_bottom.set_label('Температура (K)', fontsize=24)
        
        # ===== ПРАВЫЕ ГРАФИКИ: ДВЕ КАРТЫ =====
        ax2_top = fig.add_subplot(gs_right[0], projection=ccrs.PlateCarree())
        ax2_bottom = fig.add_subplot(gs_right[1], projection=ccrs.PlateCarree())
        
        time_window_start = matched_time - dt.timedelta(minutes=15)
        time_window_end = matched_time + dt.timedelta(minutes=15)
        
        detailed_data = all_profiles_df[
            (all_profiles_df['datetime'] >= time_window_start) & 
            (all_profiles_df['datetime'] <= time_window_end)
        ].copy()
        
        print(f"Данных в 15-минутном окне: {len(detailed_data)}")
        print(f"Временное окно: {time_window_start.strftime('%H:%M:%S')} - {time_window_end.strftime('%H:%M:%S')}")
        
        for ax, zoom, title, bounds_data in [(ax2_top, 30, 'Глобальный вид (весь пролет)', satellite_df), 
                                             (ax2_bottom, 1, 'Детальный вид (15 мин)', detailed_data)]:
            ax.add_feature(nightshade.Nightshade(matched_time, alpha=0.3, color='black'))
            
            ax.plot(sun_lon, sun_lat, 'o', color='gold', markersize=20, 
                    markeredgecolor='orange', markeredgewidth=2, transform=ccrs.PlateCarree(), zorder=5)
            
            if len(flash_times_15min) > 0:
                ax.scatter(flash_times_15min['lon'], flash_times_15min['lat'], 
                          marker='^', color='lightgray', s=80, alpha=0.8,
                          label=f'Разряды за 15 мин до ({len(flash_times_15min)})',
                          transform=ccrs.PlateCarree(), zorder=4, 
                          edgecolor='gray', linewidth=0.3)
            
            ax.add_feature(cfeature.LAND, facecolor='#e0e0e0')
            ax.add_feature(cfeature.OCEAN, facecolor='#c6e2ff')
            ax.add_feature(cfeature.COASTLINE, linewidth=0.8)
            
            gl = ax.gridlines(draw_labels=True, crs=ccrs.PlateCarree(), 
                               linewidth=0.5, color='gray', alpha=0.5, linestyle='--')
            gl.top_labels = False
            gl.right_labels = False
            
            gl.xlabel_style = {'size': 24, 'color': 'black'}  # Размер подписей долготы
            gl.ylabel_style = {'size': 24, 'color': 'black'}  # Размер подписей широты
            
            cluster_bounds = {
                'lat_min': float(data_row['lat_min']) if not isinstance(data_row['lat_min'], pd.Series) else float(data_row['lat_min'].iloc[0]),
                'lat_max': float(data_row['lat_max']) if not isinstance(data_row['lat_max'], pd.Series) else float(data_row['lat_max'].iloc[0]),
                'lon_min': float(data_row['lon_min']) if not isinstance(data_row['lon_min'], pd.Series) else float(data_row['lon_min'].iloc[0]),
                'lon_max': float(data_row['lon_max']) if not isinstance(data_row['lon_max'], pd.Series) else float(data_row['lon_max'].iloc[0])
            }
            
            rect = Rectangle(
                (cluster_bounds['lon_min'], cluster_bounds['lat_min']),
                cluster_bounds['lon_max'] - cluster_bounds['lon_min'],
                cluster_bounds['lat_max'] - cluster_bounds['lat_min'],
                facecolor='blue', alpha=0.2, edgecolor='darkblue', 
                linewidth=2, label='Зона кластера', zorder=2,
                transform=ccrs.PlateCarree()
            )
            ax.add_patch(rect)
            
            # ===== ТРАЕКТОРИЯ =====
            if zoom == 30:
                traj_data = satellite_df.copy()
            else:
                traj_data = satellite_df[
                    (satellite_df['datetime'] >= time_window_start) & 
                    (satellite_df['datetime'] <= time_window_end)
                ].copy()
            
            if len(traj_data) > 0:
                traj_unique = traj_data.groupby('datetime').agg({
                    'lat': 'mean',
                    'lon': 'mean'
                }).reset_index().sort_values('datetime')
                
                lons_fixed = traj_unique['lon'].values
                lons_fixed = np.where(lons_fixed < 0, lons_fixed + 360, lons_fixed)
                threshold = 180
                segments = []
                current_segment = []
                
                for i in range(len(lons_fixed)):
                    if len(current_segment) == 0:
                        current_segment.append(i)
                    else:
                        prev_idx = current_segment[-1]
                        if abs(lons_fixed[i] - lons_fixed[prev_idx]) > threshold:
                            segments.append(current_segment)
                            current_segment = [i]
                        else:
                            current_segment.append(i)
                if current_segment:
                    segments.append(current_segment)
                
                for seg in segments:
                    ax.plot(traj_unique['lon'].iloc[seg], traj_unique['lat'].iloc[seg], 
                            color='red', linewidth=2.5, label='Траектория' if seg is segments[0] else '',
                            alpha=0.9, zorder=3, transform=ccrs.PlateCarree())
                
                ax.plot(traj_unique['lon'].iloc[0], traj_unique['lat'].iloc[0], 'v', color='blue', 
                        markersize=12, markeredgecolor='white', markeredgewidth=1.5,
                        label='Начало', transform=ccrs.PlateCarree(), zorder=6)
                ax.plot(traj_unique['lon'].iloc[-1], traj_unique['lat'].iloc[-1], 'v', color='red', 
                        markersize=12, markeredgecolor='white', markeredgewidth=1.5,
                        label='Конец', transform=ccrs.PlateCarree(), zorder=6)
            
            ax.plot(match_point['lon'], match_point['lat'], 
                    'r*', markersize=25, markeredgecolor='yellow', markeredgewidth=1.5,
                    label='TIMED/SABER', zorder=4, transform=ccrs.PlateCarree())
            ax.plot(match_point['lon'], match_point['lat'], 
                    'ro', markersize=10, alpha=0.3, zorder=3, transform=ccrs.PlateCarree())
            
            if flash_lon is not None and flash_lat is not None:
                norm = Normalize(vmin=0, vmax=32000)
                flash_point = ax.scatter([flash_lon], [flash_lat], 
                                         c=[abs(flash_amp_value)], cmap=custom_cmap, 
                                         norm=norm, s=120, marker='^', 
                                         edgecolor='none', linewidth=2,
                                         label=f'Разряд ({flash_amp_value:.0f} A)',
                                         zorder=10, transform=ccrs.PlateCarree())
            
            # ===== ТОЧКИ ПРОФИЛЕЙ НА ГЛОБАЛЬНОЙ КАРТЕ =====
            if zoom == 30 and len(profile_points_df) > 0:
                scatter_points = ax.scatter(
                    profile_points_df['lon'], 
                    profile_points_df['lat'],
                    c=profile_points_df['median_temp'],
                    cmap=custom_cmap,
                    s=80,
                    marker='o',
                    edgecolor='black',
                    linewidth=0.5,
                    alpha=1,
                    vmin=temp_min_glob, vmax=temp_max_glob,
                    transform=ccrs.PlateCarree(),
                    zorder=15,
                    label='Профили (20 км)'
                )
                
                profile_points_df['time_diff'] = abs(profile_points_df['datetime'] - matched_time)
                center_idx = profile_points_df['time_diff'].idxmin()
                center_pos = profile_points_df.index.get_loc(center_idx)
                
                label_indices = []
                for i in range(center_pos, -1, -5):
                    label_indices.append(i)
                for i in range(center_pos + 5, len(profile_points_df), 5):
                    label_indices.append(i)
                label_indices = sorted(set(label_indices))
                
                for idx in label_indices:
                    if idx < len(profile_points_df):
                        row = profile_points_df.iloc[idx]
                        time_str = row['datetime'].strftime('%H:%M')
                        
                        if idx == center_pos:
                            bbox_props = dict(boxstyle='round,pad=0.2', facecolor='white', 
                                              edgecolor='green', linewidth=1, alpha=0.7)
                        else:
                            bbox_props = dict(boxstyle='round,pad=0.2', facecolor='white', alpha=0.7)
                        
                        ax.text(
                            row['lon'], row['lat'] + 1.5,
                            time_str,
                            fontsize=10,
                            color='black',
                            weight='normal',
                            ha='center',
                            va='bottom',
                            transform=ccrs.PlateCarree(),
                            bbox=bbox_props,
                            zorder=16
                        )

            # ===== ТОЧКИ ПРОФИЛЕЙ НА ДЕТАЛЬНОЙ КАРТЕ =====
            if zoom != 30 and len(profile_points_df) > 0:
                detailed_points = profile_points_df[
                    (profile_points_df['datetime'] >= time_window_start) & 
                    (profile_points_df['datetime'] <= time_window_end)
                ].copy()
                
                if len(detailed_points) > 0:
                    scatter_points_detailed = ax.scatter(
                        detailed_points['lon'], 
                        detailed_points['lat'],
                        c=detailed_points['median_temp'],
                        cmap=custom_cmap,
                        s=80,
                        marker='o',
                        edgecolor='black',
                        linewidth=0.5,
                        alpha=1,
                        vmin=temp_min_loc, vmax=temp_max_loc,
                        transform=ccrs.PlateCarree(),
                        zorder=15,
                        label='Профили (20 км)'
                    )
                    
                    detailed_points['time_diff'] = abs(detailed_points['datetime'] - matched_time)
                    center_idx = detailed_points['time_diff'].idxmin()
                    center_pos = detailed_points.index.get_loc(center_idx)
                    
                    label_indices = []
                    for i in range(center_pos, -1, -5):
                        label_indices.append(i)
                    for i in range(center_pos + 5, len(detailed_points), 5):
                        label_indices.append(i)
                    label_indices = sorted(set(label_indices))
                    
                    for idx in label_indices:
                        if idx < len(detailed_points):
                            row = detailed_points.iloc[idx]
                            time_str = row['datetime'].strftime('%H:%M')
                            
                            if idx == center_pos:
                                bbox_props = dict(boxstyle='round,pad=0.2', facecolor='white', 
                                                  edgecolor='green', linewidth=1, alpha=0.7)
                            else:
                                bbox_props = dict(boxstyle='round,pad=0.2', facecolor='white', alpha=0.7)
                            
                            ax.text(
                                row['lon'], row['lat'] + 5,
                                time_str,
                                fontsize=12,
                                color='black',
                                weight='normal',
                                ha='center',
                                va='bottom',
                                transform=ccrs.PlateCarree(),
                                bbox=bbox_props,
                                zorder=16
                            )
            
            if zoom == 30:
                ax.set_global()
                ax.set_title(f'{title}', fontsize=30, fontweight='bold')
            else:
                all_lons = np.concatenate([bounds_data['lon'].values, 
                                          [cluster_bounds['lon_min'], cluster_bounds['lon_max']]])
                all_lats = np.concatenate([bounds_data['lat'].values, 
                                          [cluster_bounds['lat_min'], cluster_bounds['lat_max']]])
                
                if flash_lon is not None:
                    all_lons = np.concatenate([all_lons, [flash_lon]])
                    all_lats = np.concatenate([all_lats, [flash_lat]])
                
                lon_padding = (all_lons.max() - all_lons.min()) * 0.3 * zoom
                lat_padding = (all_lats.max() - all_lats.min()) * 0.3 * zoom
                
                ax.set_xlim(all_lons.min() - lon_padding, all_lons.max() + lon_padding)
                ax.set_ylim(all_lats.min() - lat_padding, all_lats.max() + lat_padding)
                ax.set_title(f'{title}', fontsize=30, fontweight='bold')
        
        if flash_lon is not None:
            pos = ax2_bottom.get_position()
            cax = fig.add_axes([pos.x1 + 0.01, pos.y0, 0.015, pos.height])
            norm = Normalize(vmin=0, vmax=32000)
            dummy_scatter = ax2_bottom.scatter([], [], c=[], cmap=custom_cmap, norm=norm)
            cbar_flash = plt.colorbar(dummy_scatter, cax=cax)
            cbar_flash.set_label('Мощность (A)', fontsize=24)
        
        handles, labels = ax2_top.get_legend_handles_labels()
        sun_legend = [
            Patch(facecolor='black', alpha=0.3, label='Ночная сторона'),
            Line2D([0], [0], marker='o', color='gold', markersize=10, 
                   markeredgecolor='orange', linestyle='None',
                   label=f'Солнце ({sun_lat:.1f}°, {sun_lon:.1f}°)')
        ]
        
        ax2_bottom.legend(handles + sun_legend, 
                         [h.get_label() if hasattr(h, 'get_label') else h for h in handles + sun_legend],
                         loc='upper center', bbox_to_anchor=(0.5, -0.15),
                         fontsize=16, ncol=3, framealpha=1.0)
        
        if len(cluster_thunderbolts) > 0:
            cluster_start_time = cluster_thunderbolts.index.min()
            cluster_end_time = cluster_thunderbolts.index.max()
            cluster_time = f'{cluster_start_time.strftime("%H:%M:%S")} - {cluster_end_time.strftime("%H:%M:%S")}'
        else:
            cluster_time = 'нет данных'
        
        if profile_point_mode == 'single':    
            info_text = f'Интервал высот для медианных значений температур: 10 - {single_median_max} км'
        else:
            info_text = f'Интервал высот для медианных значений температур: {range_min} - {range_max} км'
        info_text += f'\nВысота пролета: {match_point["altitude_km"]:.1f} км'
        info_text += f'\nКоординаты пролета: {match_point["lat"]:.2f}°N, {match_point["lon"]:.2f}°E'
        if flash_amp_value is not None:
            info_text += f'\nМощность разряда: {flash_amp_value:.0f} A'   
        info_text += f'\nРазрядов за 15 мин до попадания: {len(flash_times_15min)}'
        info_text += f'\nВсего разрядов до попадания: {len(flash_times_all)}'
        info_text += f'\nПериод существования кластера: {cluster_time}'
        

        ax2_bottom.text(0.02, -0.1, info_text, transform=ax2_top.transAxes, fontsize=18,
                        fontweight='bold', verticalalignment='top',
                        bbox=dict(boxstyle='round,pad=0.5', facecolor='yellow', alpha=0.3))
        
        fig.suptitle(f'SABER/TIMED - {saber_file} | {date_value.strftime("%d.%m.%Y")} | Кластер #{cluster_num}', 
                     fontsize=32, fontweight='bold', y=0.98)
        
        plt.subplots_adjust(left=0.05, right=0.88, bottom=0.08, top=0.92, wspace=0.15)
        
        filename = f'SABER_cluster{cluster_num}_FULL_{date_value.strftime("%Y%m%d")}_{matched_time.strftime("%H%M%S")}_{illumination}_{altitude_rounded}.png'
        plot_path = os.path.join(output_plot_dir, filename)
        
        plt.savefig(plot_path, dpi=300, bbox_inches='tight')
        #plt.show()
        plt.close(fig)
        
        print(f"\n✓ График сохранен: {plot_path}")
        successful_clusters.append(cluster_num)
        
    except Exception as e:
        print(f"❌ ОШИБКА при обработке кластера #{cluster_num}: {e}")
        import traceback
        traceback.print_exc()
        failed_clusters.append(cluster_num)
        continue

# ============= ИТОГОВАЯ СТАТИСТИКА =============
print(f"\n{'='*60}")
print("ОБРАБОТКА ВСЕХ КЛАСТЕРОВ ЗАВЕРШЕНА")
print(f"{'='*60}")
print(f"Успешно обработано: {len(successful_clusters)} кластеров")
print(f"С ошибками: {len(failed_clusters)} кластеров")
if failed_clusters:
    print(f"Проблемные кластеры: {failed_clusters}")